# Monte Carlo Prediction: estimate a fixed policy

For a fixed policy $\pi$, Monte Carlo prediction estimates

$$V^\pi(s)=\mathbb E_\pi[G_t\mid S_t=s],\qquad
V(S_t)\leftarrow V(S_t)+\frac{G_t-V(S_t)}{N(S_t)}.$$

Here $G_t=\sum_{k=t}^{T-1}\gamma^{k-t}R_{k+1}$ is the sampled return, $T$ the episode length, $\gamma$ the discount factor, and $N(s)$ the state visit count. This notebook evaluates a uniform random policy on FrozenLake.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

ENV_ID = "FrozenLake-v1"
EPISODES = 5_000
GAMMA = 0.99
SEED = 7

env = gym.make(ENV_ID, is_slippery=True)
rng = np.random.default_rng(SEED)
state_values = np.zeros(env.observation_space.n)
visit_counts = np.zeros(env.observation_space.n, dtype=np.int64)

## 1. Sample complete episodes

The policy assigns equal probability to every action. Monte Carlo waits until an episode ends, so it uses observed returns rather than bootstrapping from current estimates.

In [ ]:
def select_action():
    return int(rng.integers(env.action_space.n))


def collect_episode():
    trajectory = []
    state, _ = env.reset()
    done = False
    while not done:
        action = select_action()
        next_state, reward, terminated, truncated, _ = env.step(action)
        trajectory.append((state, reward))
        state = next_state
        done = terminated or truncated
    return trajectory

## 2. Average first-visit returns

Compute returns backward, then update the first occurrence of each state. Both termination and truncation bound the sampled episode because this estimator has no bootstrap term.

In [ ]:
def update_values(trajectory):
    returns = np.zeros(len(trajectory))
    reward_to_go = 0.0
    for index in reversed(range(len(trajectory))):
        reward_to_go = trajectory[index][1] + GAMMA * reward_to_go
        returns[index] = reward_to_go

    seen = set()
    for index, (state, _) in enumerate(trajectory):
        if state in seen:
            continue
        seen.add(state)
        visit_counts[state] += 1
        state_values[state] += (returns[index] - state_values[state]) / visit_counts[state]


episode_returns = []
for _ in range(EPISODES):
    trajectory = collect_episode()
    update_values(trajectory)
    episode_returns.append(sum(reward for _, reward in trajectory))
env.close()

## 3. Inspect returns and values

The learning curve describes the fixed policy's sampled performance; the heatmap shows its estimated value from each state.

In [ ]:
window = 100
moving_average = np.convolve(episode_returns, np.ones(window) / window, mode="valid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(window - 1, EPISODES), moving_average)
axes[0].set(xlabel="Episode", ylabel="Return", title="Random-policy return")
image = axes[1].imshow(state_values.reshape(4, 4), cmap="viridis")
axes[1].set(title="Estimated state values")
fig.colorbar(image, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Watch the fixed policy

This opens a fresh rendered environment and samples the same uniform random policy for 5 episodes.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human", is_slippery=True)
evaluation_returns = []
try:
    for episode in range(5):
        state, _ = evaluation_env.reset(seed=10 + episode)
        total_reward = 0.0
        done = False
        while not done:
            action = int(rng.integers(evaluation_env.action_space.n))
            state, reward, terminated, truncated, _ = evaluation_env.step(action)
            total_reward += reward
            done = terminated or truncated
        evaluation_returns.append(total_reward)
finally:
    evaluation_env.close()
print("Episode returns:", evaluation_returns)